In [ ]:
from search import Stage1Search, SearchConfig, ArchParams
from CADGNCore import CADGNCore
from TokenizerFamily import TokenizerFamily
from models_config import models

search_config = SearchConfig(
    n_trials=100,
    search_epochs=15,
    full_epochs=150,
    pruning_warmup=10,
    n_startup_trials=5,
    results_dir="./search_results",
    storage="sqlite:///search.db",
    max_steps_train=550,
    max_steps_val=150,
)
do_models = ["Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]


In [ ]:
from ds.cladder import CLadderDataset, load_cladder_v1_5, CLadderLoaderConfig, CLadderSample

dsConfig = CLadderLoaderConfig(rung_filter=None, query_types=None, skip_unparseable=True)

train, vald = CLadderDataset.from_samples_split(
    samples=load_cladder_v1_5(dsConfig),
    val_size=0.2,
    stratify=True
)

In [ ]:
search = Stage1Search(
    core_factory=lambda ap: CADGNCore(
        **ap.core_kwargs(),
    ),
    family_factory=lambda ap: [
        TokenizerFamily.from_pretrained(
            model_id= k,
            max_seq_len=128,
            torch_dtype=v['dtype']
            **ap.family_kwargs(),
        )
        for k, v in models.items()
        if k in do_models
    ],
    train_dataset=train,
    val_dataset=vald,
    search_config=search_config,
)

In [ ]:
study = search.run()